In [62]:
#Librairies
import numpy as np
import matplotlib.pyplot as plt



In [63]:
#Variables

# Un mm par case de la matrice (a ajuster)
multiplicateur = 100

nb_colonnes = 5*multiplicateur
nb_lignes = 5*multiplicateur

tension_bord = 0
tension_croix = 25

#Dimension des formes de la croix

#Rectangle verticale
index_col_deb_rect_vert = 2.25*multiplicateur #index de la colonne du debut du rectangle verticale
index_col_fin_rect_vert = 2.75*multiplicateur #index de la colonne de fin du rectangle verticale

index_lig_deb_rect_vert = 1*multiplicateur #index de la ligne du debut du rectangle verticale
index_lig_fin_rect_vert = 4*multiplicateur #index de la ligne de fin du rectangle verticale

#Rectangle horizontale
index_col_deb_rect_hori = 1*multiplicateur #index de la colonne du debut du rectangle horizontale
index_col_fin_rect_hori = 4*multiplicateur #index de la colonne de fin du rectangle horizontale

index_lig_deb_rect_hori = 2.25*multiplicateur #index de la ligne du debut du rectangle horizontale
index_lig_fin_rect_hori = 2.75*multiplicateur #index de la ligne de fin du rectangle horizontale

#carré
index_col_deb_carre = 2*multiplicateur #index de la colonne du debut du carré
index_col_fin_carre = 3*multiplicateur #index de la colonne de fin du carré

index_lig_deb_carre = 2*multiplicateur #index de la ligne du debut du carré
index_lig_fin_carre = 3*multiplicateur #index de la ligne de fin du carré



In [64]:
#Creation de la matrice
matrice = np.zeros((nb_lignes, nb_colonnes))

In [65]:
#Remplissage des conducteurs de la matrice

#Boucle de parcours
for i in range (nb_colonnes):
    for j in range (nb_lignes):
        #Bord de la matrice
        if (i == nb_colonnes - 1) or  (j == nb_lignes - 1) or (i == 0) or (j == 0):
            matrice[i,j] = tension_bord
        #Rectangle verticale supperposant le carré    
        elif (i >= index_col_deb_rect_vert) and (i < index_col_fin_rect_vert) and (j >= index_lig_deb_rect_vert) and (j < index_lig_fin_rect_vert):
            matrice[i,j] = tension_croix
        #Rectangle horizontale supperposant le carré 
        elif (i >= index_col_deb_rect_hori) and (i < index_col_fin_rect_hori) and (j >= index_lig_deb_rect_hori) and (j < index_lig_fin_rect_hori):
            matrice[i,j] = tension_croix
        #Carre
        elif (i >= index_col_deb_carre) and (i < index_col_fin_carre) and (j >= index_lig_deb_carre) and (j < index_lig_fin_carre):
            matrice[i,j] = tension_croix
        else:
            matrice[i,j] = 0

In [66]:
#Visualer la matrice 

#plt.figure(figsize=(10,10))
#plt.imshow(matrice, cmap='viridis', extent=[0,nb_colonnes, nb_lignes,0])
#plt.colorbar()
#plt.xticks(np.arange(0, nb_colonnes + 1, 10))
#plt.yticks(np.arange(0, nb_lignes + 1, 10))
#plt.grid()
#plt.show()

In [67]:
#Théorie

'''
Après avoir implémenter la matrice de potentiel, l'on essaye de calculer
le champ électrique pour celui-ci

Il faut associer une case à un potentiel.

La méthode choisit est celle de la Méthode des différences finies.
L'on devrait faire la moyenne des 4 cases adjacentes

***Attention: (Elliot): J'ai demandé à l'IA, quelle méthode numérique serait le plus approprié
pour résoudre le champ électrique à partir de la matrice de potentiel. Celui-ci ma recommandé
la Méthode des différences finies. Pour parvenir à l'implémentation, je lui ai demandé des 
ressources pour m'aider à l'implémenter par moi même. Voici les ressources dont je me suis inspiré :

Video 1: https://www.youtube.com/watch?v=cwBX5gILtvs

Voici les 3 concepts que l'IA m'a propose pour etre en mesure d'obtenir un resultat coherent:
***   Les 3 concepts Python à maîtriser pour ce projet
***
***Pour comprendre parfaitement le code d'implémentation, je vous conseille de lire la documentation officielle de NumPy sur ces trois sujets précis :
***
***    Le Slicing (matrice[1:-1, 1:-1]) : C'est ce qui permet d'isoler l'intérieur de votre grille d'air sans toucher aux bords de la boîte.
***
***    L'indexation booléenne (Les "Masques") : C'est la technique (is_fixed = ...) utilisée pour figer votre électrode en croix à 25V et l'empêcher d'être modifiée par le calcul de moyenne.
***
***    np.copy() : Indispensable dans la méthode de Jacobi pour s'assurer que l'on calcule les moyennes de l'itération k en se basant uniquement sur les valeurs complètes de l'itération k−1.

'''

'\nAprès avoir implémenter la matrice de potentiel, l\'on essaye de calculer\nle champ électrique pour celui-ci\n\nIl faut associer une case à un potentiel.\n\nLa méthode choisit est celle de la Méthode des différences finies.\nL\'on devrait faire la moyenne des 4 cases adjacentes\n\n***Attention: (Elliot): J\'ai demandé à l\'IA, quelle méthode numérique serait le plus approprié\npour résoudre le champ électrique à partir de la matrice de potentiel. Celui-ci ma recommandé\nla Méthode des différences finies. Pour parvenir à l\'implémentation, je lui ai demandé des \nressources pour m\'aider à l\'implémenter par moi même. Voici les ressources dont je me suis inspiré :\n\nVideo 1: https://www.youtube.com/watch?v=cwBX5gILtvs\n\nVoici les 3 concepts que l\'IA m\'a propose pour etre en mesure d\'obtenir un resultat coherent:\n***   Les 3 concepts Python à maîtriser pour ce projet\n***\n***Pour comprendre parfaitement le code d\'implémentation, je vous conseille de lire la documentation offic

In [ ]:
erreur_tension_maximale = 0.01
erreur_tension_actuelle = 100
iteration_effectue = 0

#L'indexation booleen 
is_fixed = np.zeros_like(matrice, dtype=bool)
is_fixed[matrice == tension_croix] = True #fixe la tension de la croix
is_fixed[0, :] = True # Bord haut
is_fixed[:, 0] = True # Bord gauche
is_fixed[nb_lignes - 1, :] = True # Bord bas
is_fixed[:, nb_colonnes - 1] = True # Bord droite


#Boucle pour iterer jusqu'a obtention de l'erreur maximale autorisee
while(erreur_tension_actuelle > erreur_tension_maximale):

    #On garde la matrice precendente en copie pour la comparaison
    matrice_tension_prec = matrice.copy()

    erreur_tension_actuelle = 0 # remettre a 0 apres avoir force l'entree dans while()

    #Parcour matrice pour remplissage du champ electrique
    for i in range (1, nb_colonnes - 1):
        for j in range (1, nb_lignes - 1):

            if (is_fixed[i,j] == True):
                pass
                #on ne fait rien elle est fixe

            elif (is_fixed[i,j] == False):
                #on applique le calcul
                matrice[i,j] = 0.25 * (matrice_tension_prec[i + 1, j] + matrice_tension_prec[i - 1, j]
                                        + matrice_tension_prec[i, j + 1] + matrice_tension_prec[i, j - 1])
                difference = abs(matrice_tension_prec[i,j] - matrice[i,j])
                if (difference > erreur_tension_actuelle):
                    erreur_tension_actuelle = difference
                              

            else:
                print("Erreur inatendue dans la matrice\n\r")

    iteration_effectue += 1
    print(f"Itération {iteration_effectue} terminée | Erreur : {erreur_tension_actuelle:.4f}")

            
    


Itération 1 terminée | Erreur : 12.5000
Itération 2 terminée | Erreur : 4.6875
Itération 3 terminée | Erreur : 3.1250
Itération 4 terminée | Erreur : 2.1484
Itération 5 terminée | Erreur : 1.9531
Itération 6 terminée | Erreur : 1.4954
Itération 7 terminée | Erreur : 1.2817
Itération 8 terminée | Erreur : 1.1215
Itération 9 terminée | Erreur : 1.0254
Itération 10 terminée | Erreur : 0.8832
Itération 11 terminée | Erreur : 0.8497
Itération 12 terminée | Erreur : 0.7383
Itération 13 terminée | Erreur : 0.7067
Itération 14 terminée | Erreur : 0.6437
Itération 15 terminée | Erreur : 0.5932
Itération 16 terminée | Erreur : 0.5614
Itération 17 terminée | Erreur : 0.5357
Itération 18 terminée | Erreur : 0.4938
Itération 19 terminée | Erreur : 0.4836
Itération 20 terminée | Erreur : 0.4463
Itération 21 terminée | Erreur : 0.4367
Itération 22 terminée | Erreur : 0.4069
Itération 23 terminée | Erreur : 0.3949
Itération 24 terminée | Erreur : 0.3753
Itération 25 terminée | Erreur : 0.3580
Itératio

In [ ]:
#Attention j'ai demande a ia de faire un beau graphique

In [ ]:
# Création d'une grande figure
plt.figure(figsize=(10, 8))

# 1. Affichage de la carte de chaleur (Heatmap)
# On transpose la matrice (.T) car vos boucles utilisent i pour les colonnes (X) et j pour les lignes (Y)
img = plt.imshow(matrice.T, cmap='viridis', extent=[0, nb_colonnes, nb_lignes, 0])
plt.colorbar(img, label="Potentiel électrique (V)")

# 2. Affichage des lignes équipotentielles (Lignes de contour)
# On choisit de tracer 11 lignes entre 0V et 25V (soit une ligne tous les 2.5V)
niveaux_tension = np.linspace(0, 25, 11)

# On dessine les lignes en blanc semi-transparent par-dessus la couleur
contours = plt.contour(matrice.T, levels=niveaux_tension, colors='white', alpha=0.6, 
                       extent=[0, nb_colonnes, nb_lignes, 0])

# On ajoute les étiquettes de texte (ex: "10.0 V") directement sur les lignes
plt.clabel(contours, inline=True, fontsize=9, fmt='%1.1f V')

# 3. Esthétique (Grille et Axes)
# Pour rappel : 10 pixels = 1 mm avec votre multiplicateur
pas_mm = 10 
plt.xticks(np.arange(0, nb_colonnes + 1, pas_mm))
plt.yticks(np.arange(0, nb_lignes + 1, pas_mm))

# On cache les nombres illisibles (0 à 500) sur les axes
plt.tick_params(labelbottom=False, labelleft=False)

# Une fine grille pour se repérer
plt.grid(color='white', linestyle='-', linewidth=0.2, alpha=0.3)

plt.title(f"Distribution du potentiel après {iteration_effectue} itérations", fontsize=14, pad=15)
plt.show()